# 05 - Task 6: generation for DPO-vs-SFT comparisons
Attach outputs of notebooks 01, 02 and 03. Greedy decoding, repetition_penalty 1.1, max_new_tokens 512, identical for every model.

In [ ]:
# ---- Task 1: repository + environment setup (Kaggle, GPU T4 x2) ----
# Get the code: preferably from the private Kaggle dataset "advanced-genai-pa2-code"
# (attached as input); otherwise clone the private GitHub repo using a Kaggle secret
# named GITHUB_TOKEN (Add-ons -> Secrets).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import glob, os, shutil, subprocess, sys
if not os.path.exists(REPO):
    code = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
            if "advanced-genai-pa2-code" in f]
    if code:
        shutil.copytree(code[0], REPO)
        print("code copied from", code[0])
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
        del token, authed
os.chdir(REPO)
print(open("CODE_VERSION.txt").read() if os.path.exists("CODE_VERSION.txt") else
      subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!nvidia-smi

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
# Locate artifacts from earlier notebooks attached as input datasets.
import glob
def find_dir(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matching {pattern} under /kaggle/input - attach the previous notebook's output")
    print(pattern, "->", hits[0])
    return os.path.dirname(hits[0])

In [ ]:
SFT_BEST = find_dir("sft/best/config.json")
HH_BEST = find_dir("dpo_hh/best/config.json")
UF_BEST = find_dir("dpo_uf/best/config.json")
BASE_GENS = find_dir("generations/alpaca/base.json")  # baseline generations from notebook 01
!mkdir -p outputs/generations && cp -rn {BASE_GENS}/../* outputs/generations/ && ls -R outputs/generations

Sanity check (2-5 prompts) before the full run

In [ ]:
!python scripts/generate.py --model_path {UF_BEST} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name s --output_path /tmp/s.json --limit 3
!python scripts/generate.py --model_path {HH_BEST} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name s --output_path /tmp/s.json --limit 3

Full generation: two GPUs in parallel

In [ ]:
import subprocess
G = "python scripts/generate.py"
gpu0 = (f"{G} --device cuda:0 --model_path {SFT_BEST} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name qwen2.5-0.5b-sft --dataset_name alpaca_eval --output_path outputs/generations/alpaca/sft.json && "
        f"{G} --device cuda:0 --model_path {UF_BEST} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name qwen2.5-0.5b-dpo-uf --dataset_name alpaca_eval --output_path outputs/generations/alpaca/dpo_uf.json && "
        # alignment-tax cross-check: the safety-tuned model on general instructions
        f"{G} --device cuda:0 --model_path {HH_BEST} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name qwen2.5-0.5b-dpo-hh --dataset_name alpaca_eval --output_path outputs/generations/alpaca/dpo_hh.json")
gpu1 = (f"{G} --device cuda:1 --model_path {SFT_BEST} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name qwen2.5-0.5b-sft --output_path outputs/generations/hh/sft.json && "
        f"{G} --device cuda:1 --model_path {HH_BEST} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name qwen2.5-0.5b-dpo-hh --output_path outputs/generations/hh/dpo_hh.json && "
        # cross-check: the helpfulness-tuned model on the safety set
        f"{G} --device cuda:1 --model_path {UF_BEST} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name qwen2.5-0.5b-dpo-uf --output_path outputs/generations/hh/dpo_uf.json")
procs = [subprocess.Popen(c, shell=True, stdout=open(f"/kaggle/working/gen{i}.log", "w"), stderr=subprocess.STDOUT) for i, c in enumerate([gpu0, gpu1])]
print([p.wait() for p in procs])
!tail -2 /kaggle/working/gen0.log /kaggle/working/gen1.log

Build blinded pairwise files (randomised A/B, hidden identity metadata)

In [ ]:
!python scripts/build_pairs.py --model_path outputs/generations/alpaca/dpo_uf.json --baseline_path outputs/generations/alpaca/sft.json --eval_kind general --output_path outputs/eval/uf_dpo_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --model_path outputs/generations/hh/dpo_hh.json --baseline_path outputs/generations/hh/sft.json --eval_kind safety --output_path outputs/eval/hh_dpo_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --model_path outputs/generations/alpaca/sft.json --baseline_path outputs/generations/alpaca/base.json --eval_kind general --output_path outputs/eval/sft_vs_base_alpaca_pairs.jsonl
# optional cross-checks (alignment tax / transfer)
!python scripts/build_pairs.py --model_path outputs/generations/alpaca/dpo_hh.json --baseline_path outputs/generations/alpaca/sft.json --eval_kind general --output_path outputs/eval/xhh_dpo_vs_sft_alpaca_pairs.jsonl
!python scripts/build_pairs.py --model_path outputs/generations/hh/dpo_uf.json --baseline_path outputs/generations/hh/sft.json --eval_kind safety --output_path outputs/eval/xuf_dpo_vs_sft_hh_pairs.jsonl

Training curves for the report (SFT, both DPO runs)

In [ ]:
SFT_M = find_dir("sft/metrics.jsonl"); HH_M = find_dir("dpo_hh/metrics.jsonl"); UF_M = find_dir("dpo_uf/metrics.jsonl")
DS = find_dir("data/processed/data_stats.json")
!python scripts/plot_metrics.py --sft {SFT_M}/metrics.jsonl --dpo HH-RLHF={HH_M}/metrics.jsonl UltraFeedback={UF_M}/metrics.jsonl --out_dir report/figures
!mkdir -p outputs/run_records && for r in sft dpo_hh dpo_uf; do mkdir -p outputs/run_records/$r; done
!cp {SFT_M}/*.json {SFT_M}/metrics.jsonl outputs/run_records/sft/ ; cp {HH_M}/*.json {HH_M}/metrics.jsonl outputs/run_records/dpo_hh/ ; cp {UF_M}/*.json {UF_M}/metrics.jsonl outputs/run_records/dpo_uf/
!cp {DS}/data_stats.json outputs/run_records/ ; ls -R outputs/run_records report/figures

In [ ]:
# Small archive of metrics/configs/generations (model weights are already kept as notebook output;
# zipping them too would double disk use).
!cd {REPO} && zip -qr /kaggle/working/generation_outputs.zip outputs/generations outputs/eval outputs/run_records report/figures -x "*.safetensors" "*.bin" "*/trainer/*"
!ls -lh /kaggle/working/*.zip